# RIN - Experiment 6d: local retention plus global priors

This notebook tests whether retrieval should be normalized not only by a local retention law, but also by a query-independent global prior such as in-strength or PageRank.

The guiding architecture is:

`Score(v | q) = LocalResponse(v | q) / Prior(v)^eta`

where `LocalResponse` is either raw diffusion or the retained local winner from experiment 6c.


In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("repo root:", REPO_ROOT)


repo root: C:\Users\yaros\BlackSabbitch_repo\RIN


In [ ]:
%load_ext autoreload
%autoreload 2


## Graph definition

The graph keeps the readable diagnostic motifs from 6c and adds a new "false global fame" motif built around `espresso` versus `water`.


In [2]:
from collections import defaultdict

import numpy as np

from memory.edge import Edge
from memory.graph import MemoryGraph
from memory.node import Node

LOCAL_ALPHA = 1.0
LOCAL_BETA = 0.05
LOCAL_GAMMA = 0.5
PRIOR_ETAS = [0.0, 0.25, 0.5, 0.75, 1.0]
ESPRESSO_TAIL_WEIGHT = 0.6

NODE_SPECS = [
    ("umbrella", "concept", 0.7, 1.0, "weather"),
    ("outdoor gear", "concept", 0.7, 2.5, "weather"),
    ("rain coat", "concept", 0.7, 1.2, "weather"),
    ("boots", "concept", 0.7, 1.2, "weather"),
    ("gloves", "concept", 0.7, 1.2, "weather"),
    ("storm signal", "episode", 0.8, 1.5, "weather"),
    ("camp overview", "episode", 0.8, 1.5, "weather"),
    ("organizer pouch", "concept", 0.7, 1.0, "kit"),
    ("desk drawer", "concept", 0.7, 1.2, "kit"),
    ("cable", "concept", 0.7, 1.0, "kit"),
    ("pen", "concept", 0.7, 1.0, "kit"),
    ("coin", "concept", 0.7, 1.0, "kit"),
    ("ticket", "concept", 0.7, 1.0, "kit"),
    ("office note", "concept", 0.7, 1.0, "kit"),
    ("audit ping", "episode", 0.8, 1.5, "kit"),
    ("passport", "concept", 0.7, 1.0, "travel"),
    ("identity", "concept", 0.7, 2.5, "travel"),
    ("border ping", "episode", 0.8, 1.5, "travel"),
    ("document", "concept", 0.7, 1.2, "travel"),
    ("travel", "concept", 0.7, 2.0, "travel"),
    ("coffee", "concept", 0.7, 1.5, "health"),
    ("caffeine", "concept", 0.7, 1.0, "health"),
    ("beverage", "concept", 0.7, 2.5, "health"),
    ("energy", "concept", 0.7, 1.5, "health"),
    ("water", "concept", 0.7, 1.0, "health"),
    ("metabolic puzzle", "episode", 0.8, 1.5, "health"),
    ("morning cue", "episode", 0.8, 1.5, "morning"),
    ("wake up", "concept", 0.7, 1.2, "morning"),
    ("breakfast", "concept", 0.7, 1.2, "morning"),
    ("espresso", "concept", 0.7, 1.0, "morning"),
    ("hydration cue", "episode", 0.8, 1.5, "morning"),
    ("ocean", "concept", 0.7, 1.5, "water_fame"),
    ("rain", "concept", 0.7, 1.2, "water_fame"),
    ("river", "concept", 0.7, 1.2, "water_fame"),
    ("lake", "concept", 0.7, 1.2, "water_fame"),
    ("washing", "concept", 0.7, 1.2, "water_fame"),
    ("chemistry", "concept", 0.7, 1.2, "water_fame"),
    ("plumbing", "concept", 0.7, 1.2, "water_fame"),
    ("ice", "concept", 0.7, 1.2, "water_fame"),
    ("steam", "concept", 0.7, 1.2, "water_fame"),
    ("crema", "concept", 0.7, 1.0, "espresso_tail"),
    ("aroma", "concept", 0.7, 1.0, "espresso_tail"),
    ("barista", "concept", 0.7, 1.0, "espresso_tail"),
    ("shot", "concept", 0.7, 1.0, "espresso_tail"),
]

BIDIRECTIONAL_EDGES = [
    ("storm signal", "umbrella", 1.15),
    ("storm signal", "outdoor gear", 1.25),
    ("umbrella", "rain coat", 0.2),
    ("umbrella", "boots", 0.2),
    ("umbrella", "gloves", 0.2),
    ("outdoor gear", "rain coat", 1.0),
    ("outdoor gear", "boots", 1.0),
    ("outdoor gear", "gloves", 1.0),
    ("camp overview", "outdoor gear", 1.7),
    ("camp overview", "umbrella", 0.8),
    ("camp overview", "rain coat", 0.8),
    ("audit ping", "organizer pouch", 1.25),
    ("audit ping", "desk drawer", 1.15),
    ("organizer pouch", "cable", 0.5),
    ("organizer pouch", "pen", 0.5),
    ("organizer pouch", "coin", 0.5),
    ("organizer pouch", "ticket", 0.5),
    ("desk drawer", "office note", 2.0),
    ("border ping", "passport", 1.0),
    ("border ping", "identity", 1.0),
    ("passport", "document", 1.0),
    ("passport", "travel", 1.0),
    ("identity", "document", 1.0),
    ("identity", "travel", 1.0),
    ("metabolic puzzle", "caffeine", 1.0),
    ("metabolic puzzle", "beverage", 1.15),
    ("coffee", "caffeine", 1.5),
    ("coffee", "beverage", 1.0),
    ("coffee", "energy", 1.2),
    ("beverage", "water", 1.2),
    ("beverage", "energy", 0.6),
    ("morning cue", "wake up", 1.0),
    ("morning cue", "breakfast", 1.0),
    ("wake up", "espresso", 1.0),
    ("wake up", "water", 1.0),
    ("breakfast", "espresso", 1.0),
    ("breakfast", "water", 1.0),
    ("morning cue", "espresso", 1.2),
    ("morning cue", "water", 1.2),
    ("hydration cue", "water", 1.6),
    ("hydration cue", "espresso", 0.7),
]

DIRECTED_EDGES = [
    ("ocean", "water", 1.0),
    ("rain", "water", 1.0),
    ("river", "water", 1.0),
    ("lake", "water", 1.0),
    ("washing", "water", 1.0),
    ("chemistry", "water", 1.0),
    ("plumbing", "water", 1.0),
    ("ice", "water", 1.0),
    ("steam", "water", 1.0),
    ("ocean", "rain", 0.2),
    ("river", "ocean", 0.2),
    ("lake", "river", 0.2),
    ("ice", "lake", 0.2),
    ("steam", "rain", 0.2),
    ("espresso", "crema", ESPRESSO_TAIL_WEIGHT),
    ("espresso", "aroma", ESPRESSO_TAIL_WEIGHT),
    ("espresso", "barista", ESPRESSO_TAIL_WEIGHT),
    ("espresso", "shot", ESPRESSO_TAIL_WEIGHT),
]

QUERY_SPECS = [
    ("Q1", "what did the storm signal indicate", "storm signal", ["umbrella"], ["outdoor gear"]),
    ("Q2", "what did the audit ping indicate", "audit ping", ["organizer pouch"], ["desk drawer"]),
    ("Q3", "what did the border ping indicate", "border ping", ["passport"], ["identity"]),
    ("Q4", "tell me about the camp overview", "camp overview", ["outdoor gear"], ["umbrella", "rain coat"]),
    ("Q5", "what explained the metabolic puzzle", "metabolic puzzle", ["caffeine"], ["beverage"]),
    ("Q6", "what to drink in the morning", "morning cue", ["espresso"], ["water"]),
    ("Q7", "what did the hydration cue indicate", "hydration cue", ["water"], ["espresso"]),
]

METHOD_GRIDS = {
    "P0": [dict()],
    "P1": [dict(eta=eta) for eta in PRIOR_ETAS],
    "P2": [dict(eta=eta) for eta in PRIOR_ETAS],
    "P3": [dict(eta=eta) for eta in PRIOR_ETAS],
    "P4": [dict()],
    "P5": [dict(eta=eta) for eta in PRIOR_ETAS],
}

POSITIONS = {
    "storm signal": (-6.1, 1.9), "umbrella": (-5.0, 1.6), "outdoor gear": (-3.8, 2.0), "rain coat": (-2.8, 1.4),
    "boots": (-2.9, 0.4), "gloves": (-1.9, 0.7), "camp overview": (-4.9, 3.2),
    "audit ping": (-0.6, 4.3), "organizer pouch": (-1.6, 3.0), "desk drawer": (0.6, 3.0), "cable": (-2.4, 2.0),
    "pen": (-1.5, 1.7), "coin": (-0.7, 1.7), "ticket": (0.2, 1.8), "office note": (1.6, 2.0),
    "border ping": (3.8, 4.0), "passport": (3.0, 3.0), "identity": (4.5, 3.0), "document": (3.2, 2.0), "travel": (4.6, 2.0),
    "metabolic puzzle": (-0.2, -1.8), "coffee": (-1.5, -3.0), "caffeine": (-2.4, -3.9), "beverage": (1.0, -3.4), "energy": (-0.1, -4.4), "water": (2.5, -3.9),
    "morning cue": (0.0, -0.1), "wake up": (-1.1, 0.6), "breakfast": (1.0, 0.6), "espresso": (-0.8, -1.0), "hydration cue": (1.4, -1.0),
    "ocean": (4.7, -5.1), "rain": (3.7, -5.6), "river": (5.5, -4.8), "lake": (6.3, -5.5), "washing": (1.6, -5.3),
    "chemistry": (2.7, -5.8), "plumbing": (3.7, -6.3), "ice": (5.1, -6.1), "steam": (6.2, -4.0),
    "crema": (-2.0, -1.5), "aroma": (-1.8, -2.4), "barista": (-2.8, -0.8), "shot": (-3.0, -1.9),
}

CLUSTER_COLORS = {
    "weather": "#d6b06a",
    "kit": "#7db7d5",
    "travel": "#9fc46b",
    "health": "#d88478",
    "morning": "#c98ab2",
    "water_fame": "#75a7e3",
    "espresso_tail": "#b98f5d",
}

NODE_CAPACITY = {name: capacity for name, _kind, _conf, capacity, _cluster in NODE_SPECS}
NODE_CLUSTER = {name: cluster for name, _kind, _conf, _capacity, cluster in NODE_SPECS}


def build_graph():
    G = MemoryGraph()
    content_to_id = {}
    for i, (content, kind, conf, _capacity, _cluster) in enumerate(NODE_SPECS):
        G.add_node(Node(id=i, content=content, embedding=None, node_kind=kind, confidence=conf, provenance=[content]))
        content_to_id[content] = i
    for src, dst, weight in BIDIRECTIONAL_EDGES:
        G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type="RelatedTo", weight=weight))
        G.add_edge(Edge(src=content_to_id[dst], dst=content_to_id[src], type="RelatedTo", weight=weight))
    for src, dst, weight in DIRECTED_EDGES:
        G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type="RelatedTo", weight=weight))
    return G, content_to_id


G, content_to_id = build_graph()
ALL_EDGES = [edge for node_id in G.node_ids() for edge in G.out_edges(node_id)]
print(f"nodes: {len(G.nodes)}")
print(f"bidirectional motifs: {len(BIDIRECTIONAL_EDGES)}")
print(f"directed fame/tail edges: {len(DIRECTED_EDGES)}")


nodes: 44
bidirectional motifs: 41
directed fame/tail edges: 18


## Optional graph picture

If the plotting stack is available, draw the diagnostic world, keeping the fame edges directed.


In [3]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx
except ModuleNotFoundError as exc:
    print("graph plot skipped:", exc)
else:
    H = nx.DiGraph()
    for name, _kind, _conf, capacity, cluster in NODE_SPECS:
        H.add_node(name, capacity=capacity, cluster=cluster)
    for src, dst, weight in BIDIRECTIONAL_EDGES:
        H.add_edge(src, dst, weight=weight)
        H.add_edge(dst, src, weight=weight)
    for src, dst, weight in DIRECTED_EDGES:
        H.add_edge(src, dst, weight=weight)

    colors = [CLUSTER_COLORS[H.nodes[name]["cluster"]] for name in H.nodes]
    sizes = [460 + 160 * H.nodes[name]["capacity"] for name in H.nodes]
    widths = [0.5 + 0.8 * data["weight"] for *_edge, data in H.edges(data=True)]

    plt.figure(figsize=(13, 9))
    nx.draw_networkx_edges(H, POSITIONS, width=widths, alpha=0.35, edge_color="#666666", arrows=True, arrowsize=10)
    nx.draw_networkx_nodes(H, POSITIONS, node_color=colors, node_size=sizes, linewidths=1.0, edgecolors="#222222")
    nx.draw_networkx_labels(H, POSITIONS, font_size=8)
    plt.title("Experiment 6d: local motifs plus global fame edges")
    plt.axis("off")
    plt.show()


graph plot skipped: No module named 'matplotlib'


## Query transparency

The queries fix explicit source events. No semantic gate or anchor search is used in this benchmark.


In [4]:
for qid, query_text, source_event, correct_nodes, distractor_nodes in QUERY_SPECS:
    print(qid, query_text)
    print("  source event:", source_event)
    print("  correct:", correct_nodes)
    print("  distractors:", distractor_nodes)
    print()


Q1 what did the storm signal indicate
  source event: storm signal
  correct: ['umbrella']
  distractors: ['outdoor gear']

Q2 what did the audit ping indicate
  source event: audit ping
  correct: ['organizer pouch']
  distractors: ['desk drawer']

Q3 what did the border ping indicate
  source event: border ping
  correct: ['passport']
  distractors: ['identity']

Q4 tell me about the camp overview
  source event: camp overview
  correct: ['outdoor gear']
  distractors: ['umbrella', 'rain coat']

Q5 what explained the metabolic puzzle
  source event: metabolic puzzle
  correct: ['caffeine']
  distractors: ['beverage']

Q6 what to drink in the morning
  source event: morning cue
  correct: ['espresso']
  distractors: ['water']

Q7 what did the hydration cue indicate
  source event: hydration cue
  correct: ['water']
  distractors: ['espresso']



## Local response and global priors

`P4` and `P5` use the fixed local winner from 6c.
`P1`, `P2`, `P3`, and `P5` apply global priors with exponent `eta`.


In [5]:
def out_strength(G, node_id):
    return sum(edge.weight for edge in G.out_edges(node_id))


def in_strength(G, node_id):
    return sum(edge.weight for edge in ALL_EDGES if edge.dst == node_id)


def local_denominator(node_id):
    capacity = NODE_CAPACITY[G.nodes[node_id].content]
    s_out = max(1e-6, out_strength(G, node_id))
    return LOCAL_ALPHA + capacity + LOCAL_BETA * (s_out ** LOCAL_GAMMA)


def diffuse(G, source_node_id, use_local_retention=False, initial_activation=100.0, decay=0.8, floor_fraction=0.01, max_steps=20):
    floor = floor_fraction * initial_activation
    current = {source_node_id: initial_activation}
    peak = dict(current)

    for _step in range(max_steps):
        active = {node_id: amount for node_id, amount in current.items() if amount >= floor}
        if not active:
            break

        incoming = defaultdict(float)
        for node_id, amount in active.items():
            out_edges = G.out_edges(node_id)
            total_weight = sum(edge.weight for edge in out_edges)
            if total_weight <= 0:
                continue
            for edge in out_edges:
                incoming[edge.dst] += decay * amount * (edge.weight / total_weight)

        if use_local_retention:
            current = {dst: amount / max(1e-6, local_denominator(dst)) for dst, amount in incoming.items()}
        else:
            current = dict(incoming)

        for node_id, amount in current.items():
            if amount > peak.get(node_id, 0.0):
                peak[node_id] = amount

    return peak


def pagerank_prior(G, damping=0.85, tol=1e-10, max_iter=500):
    num_nodes = len(G.nodes)
    out_totals = np.zeros(num_nodes, dtype=np.float64)
    incoming = [[] for _ in range(num_nodes)]
    for edge in ALL_EDGES:
        out_totals[edge.src] += edge.weight
        incoming[edge.dst].append((edge.src, edge.weight))

    x = np.full(num_nodes, 1.0 / num_nodes, dtype=np.float64)
    teleport = (1.0 - damping) / num_nodes
    for _ in range(max_iter):
        sink_mass = x[out_totals == 0.0].sum()
        x_new = np.full(num_nodes, teleport + damping * sink_mass / num_nodes, dtype=np.float64)
        for dst in range(num_nodes):
            contribution = 0.0
            for src, weight in incoming[dst]:
                if out_totals[src] > 0:
                    contribution += x[src] * (weight / out_totals[src])
            x_new[dst] += damping * contribution
        if np.abs(x_new - x).sum() < tol:
            x = x_new
            break
        x = x_new

    x /= x.sum()
    return np.maximum(x.astype(np.float32), 1e-10)


OUT_PRIOR = np.array([out_strength(G, node_id) for node_id in range(len(G.nodes))], dtype=np.float32)
OUT_PRIOR = np.maximum(OUT_PRIOR / OUT_PRIOR.sum(), 1e-10)
IN_PRIOR = np.array([in_strength(G, node_id) for node_id in range(len(G.nodes))], dtype=np.float32)
IN_PRIOR = np.maximum(IN_PRIOR / IN_PRIOR.sum(), 1e-10)
PR_PRIOR = pagerank_prior(G, damping=0.85)


def prior_vector_for_method(method_name):
    if method_name == "P1":
        return OUT_PRIOR
    if method_name == "P2":
        return IN_PRIOR
    if method_name in {"P3", "P5"}:
        return PR_PRIOR
    return None


def score_map(source_event, method_name, eta=0.0):
    source_id = content_to_id[source_event]
    use_local = method_name in {"P4", "P5"}
    base = diffuse(G, source_id, use_local_retention=use_local)
    prior = prior_vector_for_method(method_name)
    scored = {}
    for node_id, raw_score in base.items():
        if prior is None:
            scored[node_id] = raw_score
        else:
            scored[node_id] = raw_score / (prior[node_id] ** eta)
    return scored


key_nodes = ["espresso", "water", "passport", "identity", "caffeine", "beverage"]
for name in key_nodes:
    node_id = content_to_id[name]
    print(
        name,
        "out_prior=", f"{OUT_PRIOR[node_id]:.6f}",
        "in_prior=", f"{IN_PRIOR[node_id]:.6f}",
        "pr=", f"{PR_PRIOR[node_id]:.6f}",
    )


espresso out_prior= 0.068108 in_prior= 0.042162 pr= 0.037337
water out_prior= 0.064865 in_prior= 0.162162 pr= 0.087651
passport out_prior= 0.032432 in_prior= 0.032432 pr= 0.032364
identity out_prior= 0.032432 in_prior= 0.032432 pr= 0.032364
caffeine out_prior= 0.027027 in_prior= 0.027027 pr= 0.026804
beverage out_prior= 0.042703 in_prior= 0.042703 pr= 0.044453


## Method landscapes

Print the score landscape for each method across the prior exponent `eta`.


In [6]:
def parameter_names_for_method(method_name):
    return ["eta"] if any("eta" in params for params in METHOD_GRIDS[method_name]) else []


def evaluate_method(method_name, params, query_specs=None):
    active_queries = QUERY_SPECS if query_specs is None else query_specs
    eta = params.get("eta", 0.0)
    solved = 0
    total_margin = 0.0
    rows = []

    for qid, query_text, source_event, correct_nodes, distractor_nodes in active_queries:
        scores = score_map(source_event, method_name, eta=eta)
        best_correct = max(scores.get(content_to_id[name], 0.0) for name in correct_nodes)
        best_distractor = max(scores.get(content_to_id[name], 0.0) for name in distractor_nodes)
        success = best_correct > best_distractor
        solved += int(success)
        total_margin += best_correct - best_distractor
        rows.append({
            "id": qid,
            "query": query_text,
            "source_event": source_event,
            "best_correct": best_correct,
            "best_distractor": best_distractor,
            "success": success,
        })

    return solved, total_margin, rows


def select_best_method_params(method_name, query_specs=None):
    best = None
    for params in METHOD_GRIDS[method_name]:
        solved, total_margin, rows = evaluate_method(method_name, params, query_specs=query_specs)
        candidate = (solved, total_margin, params, rows)
        if best is None or candidate[:2] > best[:2]:
            best = candidate
    return best


def print_rows(rows):
    for row in rows:
        print(
            f"{row['id']}  correct={row['best_correct']:.2f}  distractor={row['best_distractor']:.2f}  "
            f"success={row['success']}  source={row['source_event']}"
        )


method_landscapes = {}
for method_name, grid in METHOD_GRIDS.items():
    records = []
    for params in grid:
        solved, total_margin, _rows = evaluate_method(method_name, params)
        record = {
            "method": method_name,
            "solved": solved,
            "margin": total_margin,
            "params": dict(params),
        }
        if "eta" in params:
            record["eta"] = params["eta"]
        records.append(record)
    method_landscapes[method_name] = records

for method_name, records in method_landscapes.items():
    print(method_name, "parameter count:", len(records))
    best_records = sorted(records, key=lambda r: (r["solved"], r["margin"]), reverse=True)[:5]
    for record in best_records:
        descriptor = record["params"] if record["params"] else {"eta": 0.0}
        print("  ", descriptor, "->", f"{record['solved']}/{len(QUERY_SPECS)}", f"margin={record['margin']:.2f}")
    print()


P0 parameter count: 1
   {'eta': 0.0} -> 3/7 margin=47.54

P1 parameter count: 5
   {'eta': 0.5} -> 5/7 margin=254.17
   {'eta': 1.0} -> 4/7 margin=1405.87
   {'eta': 0.75} -> 4/7 margin=595.92
   {'eta': 0.25} -> 4/7 margin=109.33
   {'eta': 0.0} -> 3/7 margin=47.54

P2 parameter count: 5
   {'eta': 0.5} -> 6/7 margin=202.72
   {'eta': 0.25} -> 5/7 margin=94.97
   {'eta': 1.0} -> 4/7 margin=1070.05
   {'eta': 0.75} -> 4/7 margin=456.87
   {'eta': 0.0} -> 3/7 margin=47.54

P3 parameter count: 5
   {'eta': 0.5} -> 5/7 margin=206.16
   {'eta': 0.75} -> 4/7 margin=451.26
   {'eta': 0.25} -> 4/7 margin=97.36
   {'eta': 1.0} -> 3/7 margin=1017.66
   {'eta': 0.0} -> 3/7 margin=47.54

P4 parameter count: 1
   {'eta': 0.0} -> 6/7 margin=40.94

P5 parameter count: 5
   {'eta': 0.25} -> 7/7 margin=89.31
   {'eta': 0.0} -> 6/7 margin=40.94
   {'eta': 0.75} -> 5/7 margin=443.34
   {'eta': 0.5} -> 5/7 margin=197.80
   {'eta': 1.0} -> 4/7 margin=1009.86



## Optional parameter plots

Plot 1D landscapes over `eta` when `matplotlib` is available.


In [7]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    print("parameter plots skipped:", exc)
else:
    def scatter_sizes(records):
        margins = np.array([record["margin"] for record in records], dtype=np.float32)
        if len(margins) == 0 or float(margins.max() - margins.min()) < 1e-9:
            return np.full(len(records), 130.0)
        scaled = (margins - margins.min()) / (margins.max() - margins.min())
        return 90.0 + 250.0 * scaled


    def plot_method_landscape(method_name):
        records = method_landscapes[method_name]
        if not records or "eta" not in records[0]:
            record = records[0]
            print(method_name, "has no free parameters:", f"{record['solved']}/{len(QUERY_SPECS)}", f"margin={record['margin']:.2f}")
            return
        xs = [record["eta"] for record in records]
        ys = [record["margin"] for record in records]
        colors = [record["solved"] for record in records]
        sizes = scatter_sizes(records)
        fig, ax = plt.subplots(figsize=(6.2, 4.2))
        scatter = ax.scatter(xs, ys, c=colors, s=sizes, cmap="coolwarm", alpha=0.85, edgecolors="#333333")
        ax.set_xlabel("eta")
        ax.set_ylabel("margin")
        ax.set_title(f"{method_name}: prior-strength landscape")
        for record in sorted(records, key=lambda r: (r['solved'], r['margin']), reverse=True)[:5]:
            ax.annotate(f"{record['solved']}/{len(QUERY_SPECS)}", (record['eta'], record['margin']), textcoords="offset points", xytext=(5, 5), fontsize=7)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label("queries solved")
        plt.show()


    for method_name in METHOD_GRIDS:
        plot_method_landscape(method_name)


parameter plots skipped: No module named 'matplotlib'


## In-sample best setting per method

Choose the strongest setting on the full seven-query suite.


In [8]:
global_bests = {}
for method_name in METHOD_GRIDS:
    best = select_best_method_params(method_name)
    global_bests[method_name] = best
    solved, total_margin, params, rows = best
    print(method_name, solved, round(total_margin, 2), params)
    print_rows(rows)
    print()


P0 3 47.54 {}
Q1  correct=38.33  distractor=41.67  success=False  source=storm signal
Q2  correct=41.67  distractor=38.33  success=True  source=audit ping
Q3  correct=40.00  distractor=40.00  success=False  source=border ping
Q4  correct=41.21  distractor=19.39  success=True  source=camp overview
Q5  correct=37.21  distractor=42.79  success=False  source=metabolic puzzle
Q6  correct=21.82  distractor=21.82  success=False  source=morning cue
Q7  correct=55.65  distractor=24.35  success=True  source=hydration cue

P1 5 254.17 {'eta': 0.5}
Q1  correct=230.88  distractor=164.29  success=True  source=storm signal
Q2  correct=222.29  distractor=207.73  success=True  source=audit ping
Q3  correct=222.11  distractor=222.11  success=False  source=border ping
Q4  correct=162.49  distractor=131.89  success=True  source=camp overview
Q5  correct=226.34  distractor=207.07  success=True  source=metabolic puzzle
Q6  correct=83.60  distractor=85.67  success=False  source=morning cue
Q7  correct=218.51

## Leave-one-query-out validation

For each held-out query, fit the prior strength on the remaining six and evaluate on the seventh.


In [9]:
loo_summary = {}
for method_name in METHOD_GRIDS:
    held_out_solved = 0
    held_out_margin = 0.0
    chosen_params = []
    for i, held_out in enumerate(QUERY_SPECS):
        train_queries = QUERY_SPECS[:i] + QUERY_SPECS[i + 1 :]
        best_train = select_best_method_params(method_name, query_specs=train_queries)
        chosen_params.append(best_train[2])
        solved, total_margin, _rows = evaluate_method(method_name, best_train[2], query_specs=[held_out])
        held_out_solved += solved
        held_out_margin += total_margin
    loo_summary[method_name] = {
        "solved": held_out_solved,
        "margin": held_out_margin,
        "params": chosen_params,
    }
    print(method_name, held_out_solved, round(held_out_margin, 2), chosen_params)


P0 3 47.54 [{}, {}, {}, {}, {}, {}, {}]
P1 4 -32.71 [{'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 1.0}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}]
P2 6 202.72 [{'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}]
P3 4 163.33 [{'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.75}, {'eta': 0.5}, {'eta': 0.5}, {'eta': 0.5}]
P4 6 40.94 [{}, {}, {}, {}, {}, {}, {}]
P5 7 89.31 [{'eta': 0.25}, {'eta': 0.25}, {'eta': 0.25}, {'eta': 0.25}, {'eta': 0.25}, {'eta': 0.25}, {'eta': 0.25}]


## Key comparisons

Compare the best purely global method, the local-only baseline, and the hybrid winner.


In [10]:
pure_global_methods = ["P1", "P2", "P3"]
best_pure_global = max(pure_global_methods, key=lambda name: (global_bests[name][0], global_bests[name][1]))
best_hybrid = max(["P5"], key=lambda name: (loo_summary[name]["solved"], loo_summary[name]["margin"]))
local_baseline = "P4"

print("best pure-global method:", best_pure_global, global_bests[best_pure_global][2])
print_rows(global_bests[best_pure_global][3])
print()
print("local-only baseline:", local_baseline, global_bests[local_baseline][2])
print_rows(global_bests[local_baseline][3])
print()
print("hybrid winner:", best_hybrid, global_bests[best_hybrid][2])
print_rows(global_bests[best_hybrid][3])


best pure-global method: P2 {'eta': 0.5}
Q1  correct=230.88  distractor=164.29  success=True  source=storm signal
Q2  correct=222.29  distractor=207.73  success=True  source=audit ping
Q3  correct=222.11  distractor=222.11  success=False  source=border ping
Q4  correct=162.49  distractor=131.89  success=True  source=camp overview
Q5  correct=226.34  distractor=207.07  success=True  source=metabolic puzzle
Q6  correct=106.26  distractor=54.18  success=True  source=morning cue
Q7  correct=138.20  distractor=118.58  success=True  source=hydration cue

local-only baseline: P4 {}
Q1  correct=18.43  distractor=11.50  success=True  source=storm signal
Q2  correct=19.93  distractor=16.75  success=True  source=audit ping
Q3  correct=19.17  distractor=11.15  success=True  source=border ping
Q4  correct=11.38  distractor=9.32  success=True  source=camp overview
Q5  correct=17.90  distractor=11.89  success=True  source=metabolic puzzle
Q6  correct=10.26  distractor=10.28  success=False  source=mor

## Winning method details

Show the in-sample and held-out winners side by side.


In [11]:
winning_method_in_sample = max(global_bests.items(), key=lambda item: (item[1][0], item[1][1]))[0]
winning_method_held_out = max(loo_summary.items(), key=lambda item: (item[1]["solved"], item[1]["margin"]))[0]

print("winning method in-sample:", winning_method_in_sample)
print("best in-sample params:", global_bests[winning_method_in_sample][2])
print()
print_rows(global_bests[winning_method_in_sample][3])
print()
print("winning method by held-out score:", winning_method_held_out)
print("best global params for held-out winner:", global_bests[winning_method_held_out][2])
print()
print_rows(global_bests[winning_method_held_out][3])


winning method in-sample: P5
best in-sample params: {'eta': 0.25}

Q1  correct=46.05  distractor=23.64  success=True  source=storm signal
Q2  correct=40.24  distractor=37.27  success=True  source=audit ping
Q3  correct=45.20  distractor=26.29  success=True  source=border ping
Q4  correct=23.38  distractor=23.30  success=True  source=camp overview
Q5  correct=44.23  distractor=25.89  success=True  source=metabolic puzzle
Q6  correct=23.35  distractor=18.89  success=True  source=morning cue
Q7  correct=48.19  distractor=26.06  success=True  source=hydration cue

winning method by held-out score: P5
best global params for held-out winner: {'eta': 0.25}

Q1  correct=46.05  distractor=23.64  success=True  source=storm signal
Q2  correct=40.24  distractor=37.27  success=True  source=audit ping
Q3  correct=45.20  distractor=26.29  success=True  source=border ping
Q4  correct=23.38  distractor=23.30  success=True  source=camp overview
Q5  correct=44.23  distractor=25.89  success=True  source=m